# IDSW v2.0.0 Workflow: Design of Experiments (`idsw.etl.doe`)

This notebook demonstrates the **Design of Experiments (DOE)** submodule (`idsw.etl.doe`), inspired by:
- **Box, Hunter, & Hunter**: *Statistics for Experimenters* ($2^k$ factorials, Box-Cox transformations, Response Surface Methodology)
- **Barros, Scarminio, & Bruns**: *Como Fazer Experimentos* (Star / Central Composite Design)
- **Minitab & Statistica** diagnostics: Pareto of Standardized Effects, Main Effects, Interactions, Cube Plots, and 3D/Contour Response Surfaces.

In [ ]:
import numpy as np
import pandas as pd
from idsw import *

ControlVars.show_plots = True
ControlVars.show_results = True

## 1. Create Experimental Designs (Full $2^k$, Central Composite / Star, Box-Behnken)

In [ ]:
factors = {'Temperature': (120, 180), 'Pressure': (2.0, 6.0), 'Catalyst': (0.5, 1.5)}

# Generate a Central Composite / Star Design (CCD) with 3 center points
design_df, coded_df = create_factorial_design(
    factors_dict=factors,
    design_type='ccd',
    n_center_points=3,
    alpha='rotatable',
    randomize=False
)

# Simulate entering experimental yields into the 'Response' column
design_df['Response'] = (
    75.0
    + 0.25 * (design_df['Temperature'] - 150)
    + 4.0 * (design_df['Pressure'] - 4.0)
    - 0.012 * (design_df['Temperature'] - 150) ** 2
    - 1.5 * (design_df['Pressure'] - 4.0) ** 2
    + np.random.RandomState(42).normal(0, 0.8, len(design_df))
)

## 2. Factorial Analysis, Pareto of Effects, Cube Plot & Response Surface Methodology (`plotly=True`)

In [ ]:
# Analyze effects & ANOVA with interactive Plotly Pareto chart
ols_res, effects_df, anova_summ = analyze_factorial_experiment(
    df=design_df,
    factor_cols=['Temperature', 'Pressure', 'Catalyst'],
    response_col='Response',
    max_order=2,
    alpha=0.05,
    plotly=True
)

# Cube Plot on Full 2^3 Factorial
df_2k, _ = create_factorial_design(factors, design_type='full_2k', n_center_points=0, randomize=False)
df_2k['Response'] = 60.0 + 0.2 * df_2k['Temperature'] + 3.0 * df_2k['Pressure'] + 5.0 * df_2k['Catalyst']
corners_df = plot_cube_plot(
    df=df_2k,
    factor_cols=['Temperature', 'Pressure', 'Catalyst'],
    response_col='Response',
    plotly=True
)

# 3D Response Surface & 2D Contour Analysis
rsm_model, rsm_summ = response_surface_analysis(
    df=design_df,
    factor_cols=['Temperature', 'Pressure', 'Catalyst'],
    response_col='Response',
    x_factor='Temperature',
    y_factor='Pressure',
    plot_type='both',
    plotly=True
)

## 3. Box-Cox Transformation Analysis (Box, Hunter & Hunter)

In [ ]:
df_transformed, bc_summary = box_cox_transformation_analysis(
    df=design_df,
    column_to_analyze='Response',
    confidence_level=0.95,
    plotly=True
)